# 🎵 Ragam Guru — Carnatic Music AI Chatbot

> **Expert AI for Carnatic Classical Music — Built with Groq + Gradio**

This notebook runs the full **Ragam Guru** chatbot in Google Colab.

### What it does:
- 🎼 **Identify ragams** from song names (*"What ragam is Vatapi Ganapatim?"*)
- 📚 **Explain ragam details** — arohana, avarohana, mood, rasa, compositions
- 🕉️ **Teach Carnatic concepts** — Melakartha system, gamaka, tala, and more
- 💬 **Multi-turn conversation** with memory

### Before running:
1. Get a **free Groq API key** from [console.groq.com](https://console.groq.com)
2. Add it to **Colab Secrets** (🔑 key icon on left sidebar) as `GROQ_API_KEY`
3. Run all cells from top to bottom

---

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q gradio groq
print("✅ Dependencies installed!")

## Step 2: Set Up API Key

Add your Groq API key to **Colab Secrets** (🔑 icon on the left sidebar)  
Key name: `GROQ_API_KEY`

In [ ]:
import os

# Try Colab Secrets first
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("✅ API key loaded from Colab Secrets")
    else:
        raise ValueError("Empty key")
except Exception:
    # Fallback: enter manually
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your Groq API key: ")
    print("✅ API key entered manually")

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print(f"API Key set: {GROQ_API_KEY[:8]}...")

## Step 3: Load the Carnatic Ragam Knowledge Base

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CARNATIC RAGAM KNOWLEDGE BASE
# ═══════════════════════════════════════════════════════════════

RAGAM_DATABASE = {
    "shankarabharanam": {
        "name": "Shankarabharanam", "aliases": ["Bilaval", "Dheerashankarabharanam"],
        "melakartha_number": 29, "type": "Melakartha (Sampoorna)",
        "arohana": "S R2 G3 M1 P D2 N3 S", "avarohana": "S N3 D2 P M1 G3 R2 S",
        "vadi": "R2", "samvadi": "D2", "time_of_day": "Morning", "season": "All seasons",
        "mood": "Majestic, Devotional, Serene", "rasa": "Shanta, Bhakti",
        "description": "Shankarabharanam (29th Melakartha) is one of the most fundamental and celebrated ragas, equivalent to the Western major scale. It evokes grandeur, devotion, and plenitude.",
        "famous_compositions": ["Endaro Mahanubhavulu (Tyagaraja)", "Meru Samana (Tyagaraja)"],
        "famous_songs": ["Endaro Mahanubhavulu — Pancharatna Kriti"]
    },
    "kalyani": {
        "name": "Kalyani", "aliases": ["Yaman", "Mecha Kalyani"],
        "melakartha_number": 65, "type": "Melakartha (Sampoorna)",
        "arohana": "S R2 G3 M2 P D2 N3 S", "avarohana": "S N3 D2 P M2 G3 R2 S",
        "vadi": "M2", "samvadi": "S", "time_of_day": "Evening", "season": "All seasons",
        "mood": "Majestic, Romantic, Devotional", "rasa": "Shringara, Shanta",
        "description": "Kalyani (65th Melakartha) is one of the most beloved ragas. Its defining feature is Prathi Madhyamam (M2) which gives it brilliance. Equivalent to Yaman.",
        "famous_compositions": ["Nee Chesina Melu (Tyagaraja)", "Kamakshi (Dikshitar)"],
        "famous_songs": ["Nee Chesina Melu", "Kalyani Varnam"]
    },
    "bhairavi": {
        "name": "Bhairavi", "aliases": ["Sindhu Bhairavi (lighter)"],
        "melakartha_number": None, "parent_melakartha": "Natabhairavi (20)",
        "type": "Janya (Bhashanga)",
        "arohana": "S R2 G2 M1 P D1 N2 S", "avarohana": "S N2 D1 P M1 G2 R1 S",
        "vadi": "G2", "samvadi": "N2", "time_of_day": "Early morning", "season": "Winter",
        "mood": "Deeply melancholic, devotional, full of pathos", "rasa": "Karuna, Bhakti",
        "description": "Bhairavi is one of the most expressive and emotionally rich ragas, traditionally sung at the end of concerts. Evokes longing, devotion, and deep pathos.",
        "famous_compositions": ["Viriboni Varnam", "Pahimam Sri (Tyagaraja)"],
        "famous_songs": ["Viriboni Varnam", "Kurai Ondrum Illai (devotional)"]
    },
    "mohanam": {
        "name": "Mohanam", "aliases": ["Bhoop", "Pentatonic major"],
        "melakartha_number": None, "parent_melakartha": "Harikambhoji (28)",
        "type": "Janya (Audava-Audava)",
        "arohana": "S R2 G3 P D2 S", "avarohana": "S D2 P G3 R2 S",
        "vadi": "G3", "samvadi": "D2", "time_of_day": "Evening", "season": "Spring",
        "mood": "Joyful, bright, romantic, charming", "rasa": "Shringara, Hasya",
        "description": "Mohanam is a pentatonic raga (5 notes, no Ma/Ni) — bright, uplifting, and universally appealing. Equivalent to Bhupali in Hindustani music.",
        "famous_compositions": ["Mohanara Mahimaalo (Tyagaraja)"],
        "famous_songs": ["Vande Mataram (intro)", "Kaadhal Rojave (ARR)"]
    },
    "hamsadhwani": {
        "name": "Hamsadhwani", "aliases": [],
        "melakartha_number": None, "parent_melakartha": "Kalyani (65)",
        "type": "Janya (Audava-Audava)",
        "arohana": "S R2 G3 P N3 S", "avarohana": "S N3 P G3 R2 S",
        "vadi": "G3", "samvadi": "N3", "time_of_day": "Night (auspicious)", "season": "All seasons",
        "mood": "Auspicious, serene, joyful, divine", "rasa": "Bhakti, Shanta",
        "description": "Hamsadhwani ('sound of the swan') — a pentatonic raga sung at auspicious beginnings. Home of the iconic 'Vatapi Ganapatim'.",
        "famous_compositions": ["Vatapi Ganapatim Bhaje (Dikshitar)", "Raghuvamsa Sudha (Tyagaraja)"],
        "famous_songs": ["Vatapi Ganapatim"]
    },
    "hindolam": {
        "name": "Hindolam", "aliases": ["Malkauns", "Minor pentatonic"],
        "melakartha_number": None, "parent_melakartha": "Todi (45)",
        "type": "Janya (Audava-Audava)",
        "arohana": "S G2 M1 D1 N2 S", "avarohana": "S N2 D1 M1 G2 S",
        "vadi": "M1", "samvadi": "S", "time_of_day": "Late night", "season": "Winter",
        "mood": "Melancholic, mysterious, haunting, meditative", "rasa": "Karuna, Shringara",
        "description": "Hindolam — no Ri or Pa — has a deeply haunting and melancholic character. Equivalent to Malkauns. A powerful late-night raga.",
        "famous_compositions": ["Chakkani Raja (Tyagaraja)"],
        "famous_songs": ["Enna Thavam Seydhanai (M.S. Subbulakshmi)"]
    },
    "kiravani": {
        "name": "Kiravani", "aliases": ["Keeravani", "Harmonic minor"],
        "melakartha_number": 21, "type": "Melakartha",
        "arohana": "S R2 G2 M1 P D1 N3 S", "avarohana": "S N3 D1 P M1 G2 R2 S",
        "vadi": "S", "samvadi": "P", "time_of_day": "Night", "season": "Monsoon",
        "mood": "Melancholic, yearning, deeply emotional", "rasa": "Karuna",
        "description": "Kiravani (21st Melakartha) = harmonic minor. The augmented second D1→N3 creates intense yearning. Very popular in South Indian film music.",
        "famous_compositions": ["Rupamu Juchi (Tyagaraja)"],
        "famous_songs": ["Ye Maya Chesave (film)"]
    },
    "sri": {
        "name": "Sri", "aliases": ["Sri Ragam"],
        "melakartha_number": None, "parent_melakartha": "Kharaharapriya (22)",
        "type": "Janya (Vakra)",
        "arohana": "S R2 M1 P N2 S", "avarohana": "S N2 P M1 R2 G2 R2 S",
        "vadi": "R2", "samvadi": "P", "time_of_day": "Evening", "season": "All seasons",
        "mood": "Devotional, auspicious, majestic", "rasa": "Bhakti, Shanta",
        "description": "Sri Ragam — auspicious raga associated with goddess Lakshmi. Home of Tyagaraja's Pancharatna Kriti 'Entharo Mahanubhavulu'.",
        "famous_compositions": ["Entharo Mahanubhavulu (Tyagaraja)"],
        "famous_songs": ["Entharo Mahanubhavulu"]
    },
    "todi": {
        "name": "Todi", "aliases": ["Subhapantuvarali"],
        "melakartha_number": 45, "type": "Melakartha (Sampoorna)",
        "arohana": "S R1 G2 M2 P D1 N2 S", "avarohana": "S N2 D1 P M2 G2 R1 S",
        "vadi": "G2", "samvadi": "N2", "time_of_day": "Morning", "season": "All seasons",
        "mood": "Profoundly devotional, melancholic, moving", "rasa": "Karuna, Bhakti",
        "description": "Todi (45th Melakartha) — a master musician's raga. The combination of komal and tivra swaras creates uniquely intense and plaintive character.",
        "famous_compositions": ["Ninnukori (Tyagaraja)"],
        "famous_songs": []
    },
    "nilambari": {
        "name": "Nilambari", "aliases": [],
        "melakartha_number": None, "parent_melakartha": "Harikambhoji (28)",
        "type": "Janya",
        "arohana": "S R2 G3 M1 P N2 S", "avarohana": "S N2 P M1 G3 R2 S",
        "vadi": "G3", "samvadi": "N2", "time_of_day": "Night", "season": "All seasons",
        "mood": "Soothing, tender, dreamy, lullaby", "rasa": "Shanta, Vatsalya",
        "description": "Nilambari — universally associated with lullabies. Captures the stillness of night. Perfect for calming children to sleep.",
        "famous_compositions": ["Thaye Yasoda (lullaby)"],
        "famous_songs": ["Thaye Yasoda", "Nila Kayuthu (film)"]
    },
}

SONG_TO_RAGAM = {
    "entharo mahanubhavulu": "sri",
    "endaro mahanubhavulu": "sri",
    "vatapi ganapatim": "hamsadhwani",
    "vatapi ganapatim bhaje": "hamsadhwani",
    "viriboni": "bhairavi",
    "nidhi chala sukhama": "bilahari",
    "nagumomu ganaleni": "abhogi",
    "kaadhal rojave": "mohanam",
    "vellai pookkal": "sindhubhairavi",
    "enna thavam seydhanai": "hindolam",
    "ye maya chesave": "kiravani",
    "nila kayuthu": "nilambari",
    "thaye yasoda": "nilambari",
    "koluvaiyunnade": "reethigowla",
}

def get_ragam_info(name):
    key = name.lower().strip().replace(' ', '_')
    if key in RAGAM_DATABASE:
        return RAGAM_DATABASE[key]
    for k, v in RAGAM_DATABASE.items():
        if name.lower() in v['name'].lower() or v['name'].lower() in name.lower():
            return v
    return None

def detect_from_song(song):
    s = song.lower().strip()
    for k, rk in SONG_TO_RAGAM.items():
        if k in s or s in k:
            return RAGAM_DATABASE.get(rk)
    return None

def format_ragam(r):
    if not r:
        return ""
    return (
        f"Ragam: {r['name']} | Type: {r.get('type','N/A')} | "
        f"Arohana: {r.get('arohana','N/A')} | Avarohana: {r.get('avarohana','N/A')} | "
        f"Vadi: {r.get('vadi','N/A')} | Mood: {r.get('mood','N/A')} | "
        f"Rasa: {r.get('rasa','N/A')} | Time: {r.get('time_of_day','N/A')} | "
        f"Description: {r.get('description','N/A')} | "
        f"Famous Compositions: {'; '.join(r.get('famous_compositions',[])) or 'None'}"
    )

def build_context():
    return '\n'.join(
        f"- {v['name']}: {v.get('arohana','N/A')} | {v.get('mood','')}"
        for v in RAGAM_DATABASE.values()
    )

print(f"✅ Loaded {len(RAGAM_DATABASE)} ragams into memory!")
print("Ragams:", ', '.join(v['name'] for v in RAGAM_DATABASE.values()))

## Step 4: Set Up the AI System

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
MODEL = "llama-3.3-70b-versatile"

RAGAM_CONTEXT = build_context()
ALL_NAMES = ', '.join(v['name'] for v in RAGAM_DATABASE.values())

SYSTEM_PROMPT = f"""You are **Ragam Guru**, an expert AI assistant specializing in Carnatic classical music.

You deeply understand:
- All 72 Melakartha ragas and hundreds of Janya ragas
- Arohana (ascending scale) and Avarohana (descending scale) of every raga
- Vadi, Samvadi, Gamaka, and characteristic phrases (prayogas)
- Time of day, season, mood, and rasa of each raga
- Famous compositions by Tyagaraja, Muthuswami Dikshitar, Shyama Shastri, and others
- The Melakartha system — 72 parent scales of Carnatic music
- Carnatic music concepts: tala, shruti, gamaka, alapana, neraval, swaraprastara

## YOUR KNOWLEDGE BASE
{RAGAM_CONTEXT}

## RAGAS AVAILABLE: {ALL_NAMES}

## HOW TO RESPOND
When asked about a RAGAM — provide: full name, arohana, avarohana, vadi/samvadi, mood, rasa, time, season, famous compositions.
When asked to IDENTIFY a raga from a song — identify it, explain why, give other songs in the same raga.
When asked CONCEPTUAL questions — give clear, scholarly, accessible explanations.

## TONE
Friendly, scholarly, enthusiastic about Carnatic music. Use proper terminology.
Use swara notation: S R1/R2/R3 G1/G2/G3 M1/M2 P D1/D2/D3 N1/N2/N3
"""

print("✅ AI system configured with LLaMA 3.3 70B via Groq!")

## Step 5: Launch the Chatbot Interface

> A **public Gradio link** will be generated. Share it with anyone!

In [ ]:
import gradio as gr

def extract_context(message):
    """Inject ragam data into the message if a known song/raga is detected."""
    r = detect_from_song(message)
    if r:
        return f"\n\n[CONTEXT FROM DATABASE]\n{format_ragam(r)}\n"
    for key, ragam in RAGAM_DATABASE.items():
        if ragam['name'].lower() in message.lower():
            return f"\n\n[CONTEXT FROM DATABASE]\n{format_ragam(ragam)}\n"
    return ""

def chat(message, history):
    """Streaming chat function."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in (history or []):
        messages.append({"role": turn["role"], "content": turn["content"]})
    enriched = message + extract_context(message)
    messages.append({"role": "user", "content": enriched})
    stream = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=1024, temperature=0.7, stream=True
    )
    response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response

# ── Gradio UI ──────────────────────────────────────────────────────────
EXAMPLES = [
    ["What ragam is Entharo Mahanubhavulu?"],
    ["Tell me everything about Kalyani"],
    ["What ragam is Vatapi Ganapatim?"],
    ["Difference between Bhairavi and Sindhu Bhairavi?"],
    ["Which ragas are sung at early morning?"],
    ["What is the Melakartha system?"],
    ["Which raga is used for lullabies?"],
    ["List ragas that evoke sadness"],
    ["Tell me about Hamsadhwani"],
    ["How does gamaka work in Carnatic music?"],
]

THEME = gr.themes.Soft(
    primary_hue="orange", secondary_hue="amber", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
)

with gr.Blocks(theme=THEME, title="Ragam Guru — Carnatic Music AI", css="""
    .ragam-header { background: linear-gradient(135deg,#1a0a00,#2d1200,#1a0a00);
        border:1px solid #FF6B35; border-radius:16px; padding:24px 28px; margin-bottom:16px; }
    .ragam-header h1 { color:#FF6B35; font-size:2em; margin:0 0 6px; font-weight:800; }
    .ragam-header p { color:#c9a882; margin:0; line-height:1.6; }
""") as demo:
    gr.HTML("""
    <div class="ragam-header">
        <h1>🎵 Ragam Guru</h1>
        <p>Expert AI for <strong>Carnatic Classical Music</strong> — powered by Groq · LLaMA 3.3 70B</p>
        <p style="color:#FFD700; margin-top:8px; font-size:0.95em;">
        🎼 Identify ragas · 📚 Learn details · 🕉️ Explore concepts
        </p>
    </div>
    """)

    chatbot = gr.Chatbot(
        elem_id="chatbot", label="Ragam Guru", height=500,
        type="messages", show_copy_button=True,
        placeholder="<div style='text-align:center;padding:40px;color:#555'>"
                    "<div style='font-size:2.5em'>🎵</div>"
                    "<div>Ask about any ragam or song!</div></div>"
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask about any ragam, song, or Carnatic concept...",
            show_label=False, scale=9, lines=1
        )
        btn = gr.Button("Ask 🎵", scale=1, variant="primary")

    clear = gr.Button("🗑️ Clear Chat", size="sm")

    gr.HTML('<div style="color:#FF6B35;font-weight:700;font-size:0.9em;margin:12px 0 6px;">💡 Try these questions</div>')
    gr.Examples(examples=EXAMPLES, inputs=msg, label="")

    with gr.Accordion("ℹ️ About", open=False):
        gr.Markdown(f"""
        **Ragam Guru** has {len(RAGAM_DATABASE)} ragas in its database.  
        Model: LLaMA 3.3 70B · Groq API · Gradio  
        Built for Carnatic music enthusiasts worldwide 🎵
        """)

    def user_msg(message, history):
        history = history or []
        history.append({"role": "user", "content": message})
        return "", history

    def bot_resp(history):
        if not history:
            return history
        last = history[-1]["content"]
        history.append({"role": "assistant", "content": ""})
        for p in chat(last, history[:-1]):
            history[-1]["content"] = p
            yield history

    msg.submit(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(bot_resp, chatbot, chatbot)
    btn.click(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(bot_resp, chatbot, chatbot)
    clear.click(lambda: [], None, chatbot)

print("🚀 Launching Ragam Guru...")
demo.queue()
demo.launch(share=True)  # share=True generates a public link in Colab

---
## 📝 Notes & Tips

- The **public Gradio link** above expires after 72 hours (Colab session limit)
- For a **permanent deployment**, push to GitHub and host on [Render](https://render.com)
- See the [README](https://github.com/YOUR_USERNAME/carnatic-ragam-ai/blob/main/README.md) for full deployment instructions
- Get your free Groq API key at [console.groq.com](https://console.groq.com)

### Example Questions to Try:
| Ask | Expected response |
|---|---|
| `What ragam is Entharo Mahanubhavulu?` | Sri Ragam details |
| `Tell me about Kalyani` | 65th Melakartha, M2, evening raga... |
| `Which ragas evoke sadness?` | Bhairavi, Hindolam, Sahana... |
| `What is the Melakartha system?` | 72 parent scales explained |
| `What ragam is Vatapi Ganapatim?` | Hamsadhwani details |
